# Strands Agent Deployment with Observability — FSI Edition

This lab demonstrates deploying a Strands Agent to Bedrock AgentCore Runtime with full observability — critical for FSI compliance and audit trails.

## Overview

In this lab, you will:
- Deploy a Strands Agent with FSI tools to AgentCore Runtime
- Invoke the deployed agent via boto3
- View traces and spans in CloudWatch (GenAI Observability)
- Understand how observability supports FSI compliance

## Why Observability for FSI?

Regulators require financial institutions to:
- **Audit every decision** — What did the agent do and why?
- **Trace data flow** — Where did the data come from?
- **Monitor performance** — Is the agent responding within SLAs?
- **Detect anomalies** — Is the agent behaving unexpectedly?

## Prerequisites

⚠️ **Important**: Enable [CloudWatch Transaction Search](https://console.aws.amazon.com/cloudwatch/home#logsV2:transaction-search) and set X-Ray trace indexing to **100%** before starting this lab.

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

In [16]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## Step 1: Create the Agent Application

We'll create a Python file that defines our FSI agent with tools, wrapped in the `BedrockAgentCoreApp` class for deployment.

In [21]:
%%writefile strands_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Setup Nova Pro model ID based on AWS region
NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
region = boto3.session.Session().region_name
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

@tool
def validate_transaction(amount: float, merchant_category: str = "general") -> str:
    '''Validate a transaction against risk rules.
    Args:
        amount: Transaction amount in AUD
        merchant_category: Category (general, crypto, gambling, high_risk)
    '''
    risk_score = 0
    flags = []
    if amount > 10000: flags.append("HIGH_VALUE"); risk_score += 3
    if merchant_category in ("crypto", "gambling"): flags.append(f"HIGH_RISK_{merchant_category.upper()}"); risk_score += 4
    decision = "BLOCKED" if risk_score >= 7 else "REVIEW" if risk_score >= 4 else "APPROVED"
    return f"Decision: {decision} | Risk: {risk_score}/10 | Flags: {flags}"

@tool
def get_account_balance(account_id: str) -> str:
    '''Get account balance.
    Args:
        account_id: Account identifier
    '''
    balances = {"ACC-001": "$125,430.50", "ACC-002": "$2,340,000.00", "ACC-003": "$45,200.75"}
    return f"Account {account_id}: Balance {balances.get(account_id, 'Not found')}"

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt="You are a banking operations assistant. Validate transactions and check accounts. Be concise.",
    tools=[validate_transaction, get_account_balance, calculator],
)

@app.entrypoint
async def fsi_agent(payload, context):
    """Invoke the FSI agent with a payload"""
    user_input = payload.get("prompt", "No prompt found")
    response = agent(user_input)
    return response

if __name__ == "__main__":
    app.run()



Overwriting strands_agent.py


In [22]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore

Overwriting requirements.txt


## Step 2: Deploy to AgentCore Runtime

In [23]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name

agentcore_runtime = Runtime()

print("Configuring and deploying FSI agent...")
response = agentcore_runtime.configure(
    entrypoint="strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="fsi_observability_agent",
    non_interactive=True,
)
print("Configuration completed")

print("Launching deployment (3-5 minutes)...")
launch_result = agentcore_runtime.launch()
runtime_id = launch_result.agent_id
runtime_arn = launch_result.agent_arn
print(f"Deployed! Runtime ID: {runtime_id}")
print(f"ARN: {runtime_arn}")


Entrypoint parsed: file=/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/05-agentcore-runtime-observability-fsi/strands_agent.py, bedrock_agentcore_name=strands_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: fsi_observability_agent


Configuring and deploying FSI agent...


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/zohaibso/AI 
Workshops/FSI-AgentCore-Workshop/05-agentcore-runtime-observability-fsi/Dockerfile

Generated .dockerignore: /Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/05-agentcore-runtime-observability-fsi/.dockerignore
Keeping 'fsi_observability_agent' as default agent
Bedrock AgentCore configured: /Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/05-agentcore-runtime-observability-fsi/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'fsi_observability_agent' to account 905035168378 (ap-southeast-2)
Generated image tag: 20260601-041247-575
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: fsi_observability_agent
ECR repository available: 9050

Configuration completed
Launching deployment (3-5 minutes)...
✅ Reusing existing ECR repository: 905035168378.dkr.ecr.ap-southeast-2.amazonaws.com/bedrock-agentcore-fsi_observability_agent


✅ Reusing existing execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-0bfe1849c6
Execution role available: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKRuntime-ap-southeast-2-0bfe1849c6
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: fsi_observability_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-0bfe1849c6
Reusing existing CodeBuild execution role: arn:aws:iam::905035168378:role/AmazonBedrockAgentCoreSDKCodeBuild-ap-southeast-2-0bfe1849c6
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: fsi_observability_agent/source.zip
Updated CodeBuild project: bedrock-agentcore-fsi_observability_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 5.2s
🔄 DOW

Deployed! Runtime ID: fsi_observability_agent-LdXTFc3dIy
ARN: arn:aws:bedrock-agentcore:ap-southeast-2:905035168378:runtime/fsi_observability_agent-LdXTFc3dIy


## Step 3: Invoke the Deployed Agent

In [24]:
import boto3
import json
import uuid

SESSION_ID = str(uuid.uuid4())
PROMPT = "Validate a 20000 dollar transaction to a crypto exchange and check balance for ACC-001"

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID,
    payload=json.dumps({"prompt": PROMPT})
)

if "text/event-stream" in boto3_response.get("contentType", ""):
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            print(line.decode("utf-8") if isinstance(line, bytes) else line)
else:
    print(boto3_response)


{'ResponseMetadata': {'RequestId': 'd36e4355-6b81-4bcd-a330-5e3cd13d8a52', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 01 Jun 2026 04:13:47 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': 'd36e4355-6b81-4bcd-a330-5e3cd13d8a52', 'x-amzn-bedrock-agentcore-runtime-session-id': 'd443c7ef-e3a4-4716-afa2-c7238f71cf16'}, 'RetryAttempts': 0}, 'runtimeSessionId': 'd443c7ef-e3a4-4716-afa2-c7238f71cf16', 'contentType': 'application/json', 'statusCode': 200, 'response': <botocore.response.StreamingBody object at 0x11b055240>}


## Step 4: View Traces in CloudWatch

Navigate to the [CloudWatch Console → Application Signals → Traces](https://console.aws.amazon.com/cloudwatch/home#xray:traces) to see:

- **End-to-end trace** of the agent invocation
- **Spans** for each tool call (validate_transaction, get_account_balance)
- **Latency** breakdown per component
- **Model invocation** details (tokens, duration)

You can also check the **AgentCore tab** in CloudWatch for a fleet-level view.

### What the Audit Trail Shows (FSI Compliance)

| Trace Element | Compliance Value |
|--------------|------------------|
| Request timestamp | When was the decision made? |
| Tool calls | What data was consulted? |
| Model reasoning | Why was this decision reached? |
| Response | What was communicated? |
| Latency | Was it within SLA? |

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up
# agentcore_runtime.delete()
# print("✅ Runtime deleted")


## Summary

- ✅ Deployed FSI agent to AgentCore Runtime
- ✅ Invoked via boto3 with IAM authentication
- ✅ Viewed traces in CloudWatch (full audit trail)
- ✅ Understood observability for FSI compliance

### Next: Lab 06 — Memory
We'll add persistent memory so the agent remembers client context across sessions.